**Modelo clasificador de imagenes generadas con DeepFake vs Reales**

*Dataset obtenido de:* https://www.kaggle.com/datasets/muhammadbilal6305/200k-real-vs-ai-visuals-by-mbilal/data

In [ ]:
import os
from google.colab import drive

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
else:
    print("Drive ya estaba montado.")

In [ ]:
dataset_output_dir = "/content/drive/MyDrive/gravex200ds"
class_names = ['ai_images', 'real']

def find_class_dirs(base_dir, class_names):
    """Busca recursivamente la carpeta que contiene las subcarpetas de clase esperadas."""
    for root, dirs, files in os.walk(base_dir):
        if all(c in dirs for c in class_names):
            return root
    return None

output_dir_has_content = os.path.isdir(dataset_output_dir) and len(os.listdir(dataset_output_dir)) > 0

if output_dir_has_content:
    print("output_dir ya tiene contenido -> se omite kagglehub.dataset_download()")
    found_dir = find_class_dirs(dataset_output_dir, class_names)
    if found_dir is None:
        raise RuntimeError(
            f"{dataset_output_dir} no está vacío pero no se encontraron las carpetas "
            f"{class_names} dentro de ningún subdirectorio. Revisa manualmente el contenido."
        )
    images_dir_path = found_dir + "/"
else:
    import kagglehub
    path = kagglehub.dataset_download(
        "muhammadbilal6305/200k-real-vs-ai-visuals-by-mbilal",
        output_dir=dataset_output_dir
    )
    print("Path to dataset files:", path)
    found_dir = find_class_dirs(path, class_names)
    assert found_dir is not None, f"No se encontraron las carpetas {class_names} tras la descarga en {path}"
    images_dir_path = found_dir + "/"

print("images_dir_path resuelto:", images_dir_path)

for c in class_names:
    n = len(os.listdir(os.path.join(images_dir_path, c)))
    print(f"{c}: {n} archivos")
    assert n > 0, f"Carpeta '{c}' vacía"

**Preprocesamiento**

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications.resnet50 import preprocess_input

# ============== CONSTANTES ==============

IMG_SIZE = (224, 224)
BATCH_SIZE = 64
SEED = 123
VAL_SPLIT = 0.2

train_paths_by_class = [[], []]
val_paths, val_labels = [], []
rng = np.random.default_rng(SEED)

for idx, class_name in enumerate(class_names):
    class_dir = os.path.join(images_dir_path, class_name)
    files = np.array([os.path.join(class_dir, f) for f in os.listdir(class_dir)])
    rng.shuffle(files)
    n_val = int(len(files) * VAL_SPLIT)
    val_paths.extend(files[:n_val]); val_labels.extend([idx] * n_val)
    train_paths_by_class[idx] = files[n_val:].tolist()

print("Train por clase:", [len(p) for p in train_paths_by_class])
print("Val:  ", np.bincount(val_labels))


# ============== AUGMENTATION (solo train) ==============

def augment_image(img: tf.Tensor) -> tf.Tensor:
    """Aplica variabilidad realista al estilo de las condiciones que el
    modelo enfrenta en producción: compresión variable, encuadre imperfecto,
    iluminación diversa. Espera img en rango [0, 255] float32, ANTES de
    preprocess_input."""

    # --- Geometría ---
    img = tf.image.random_flip_left_right(img)

    # Zoom / recorte variable: simula que el margen de smart_crop no
    # siempre es idéntico (se ajusta dinámicamente según rostros vecinos)
    zoom_factor = tf.random.uniform([], 0.85, 1.15)
    h, w = IMG_SIZE
    new_h = tf.cast(tf.cast(h, tf.float32) * zoom_factor, tf.int32)
    new_w = tf.cast(tf.cast(w, tf.float32) * zoom_factor, tf.int32)
    img = tf.image.resize(img, (new_h, new_w))
    img = tf.image.resize_with_crop_or_pad(img, h, w)

    # Traslación leve: simula que el detector no siempre centra el rostro
    # perfectamente dentro de la caja detectada
    max_shift = 15
    dx = tf.random.uniform([], -max_shift, max_shift, dtype=tf.int32)
    dy = tf.random.uniform([], -max_shift, max_shift, dtype=tf.int32)
    img = tf.roll(img, shift=[dy, dx], axis=[0, 1])

    # --- Fotometría ---
    img = tf.image.random_brightness(img, max_delta=25)
    img = tf.image.random_contrast(img, lower=0.85, upper=1.15)

    # Calidad JPEG variable: ataca directamente la dependencia a firmas
    # de compresión específicas del dataset de entrenamiento
    quality = tf.random.uniform([], 40, 100, dtype=tf.int32)
    img_uint8 = tf.cast(tf.clip_by_value(img, 0, 255), tf.uint8)
    img = tf.cast(tf.image.adjust_jpeg_quality(img_uint8, quality), tf.float32)

    # Blur ocasional leve: simula fotos de menor resolución/calidad
    if tf.random.uniform([]) < 0.3:
        img = tf.nn.avg_pool2d(img[tf.newaxis, ...], ksize=3, strides=1, padding='SAME')[0]

    img = tf.clip_by_value(img, 0.0, 255.0)
    return img


# ============== PIPELINES DE CARGA ==============

def load_image_train(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    img = augment_image(img)      # <- solo en entrenamiento
    img = preprocess_input(img)   # <- después del augmentation, nunca antes
    return img, label

def load_image_val(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    img = preprocess_input(img)   # sin augmentation -- validación debe ser determinista
    return img, label


def make_class_dataset(paths, label):
    ds = tf.data.Dataset.from_tensor_slices((paths, [label] * len(paths)))
    ds = ds.shuffle(buffer_size=len(paths), seed=SEED)
    ds = ds.map(load_image_train, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.repeat()  # necesario: sample_from_datasets consume streams infinitos

class_datasets = [
    make_class_dataset(train_paths_by_class[idx], idx)
    for idx in range(len(class_names))
]

steps_per_epoch = sum(len(p) for p in train_paths_by_class) // BATCH_SIZE

train_ds = tf.data.Dataset.sample_from_datasets(
    class_datasets, weights=[0.5, 0.5], seed=SEED
).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

def make_val_dataset(paths, labels):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(load_image_val, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_ds = make_val_dataset(val_paths, val_labels)

In [ ]:
import matplotlib.pyplot as plt

sample_path = train_paths_by_class[0][0]  # cualquier imagen de ejemplo
raw = tf.io.read_file(sample_path)
raw = tf.image.decode_image(raw, channels=3, expand_animations=False)
raw = tf.image.resize(raw, IMG_SIZE)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax in axes.flat:
    augmented = augment_image(raw)
    ax.imshow(tf.clip_by_value(augmented, 0, 255).numpy().astype("uint8"))
    ax.axis("off")
plt.tight_layout()
plt.show()

**Entrenamiento**

In [ ]:
import json
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

EPOCHS_TRAIN_1 = 10
EPOCHS_TRAIN_2 = 100
PATIENCE = 3

CKPT_DIR = "/content/drive/MyDrive/ai_vs_real_model"
STATE_PATH = os.path.join(CKPT_DIR, "training_state_v5.json")
LATEST_MODEL_PATH = os.path.join(CKPT_DIR, "latest_model_v5.keras")
BEST_MODEL_PATH = os.path.join(CKPT_DIR, "best_model_v5.keras")
os.makedirs(CKPT_DIR, exist_ok=True)

def load_state():
    if os.path.exists(STATE_PATH):
        with open(STATE_PATH) as f:
            return json.load(f)
    return {"phase": 1, "next_epoch": 0, "best_val_accuracy": 0.0}

def save_state(phase, next_epoch, best_val_accuracy):
    with open(STATE_PATH, "w") as f:
        json.dump({"phase": phase, "next_epoch": next_epoch,
                    "best_val_accuracy": float(best_val_accuracy)}, f)

class StateCheckpoint(keras.callbacks.Callback):
    """Guarda el modelo completo cada época (para reanudar) y actualiza
    best_model.keras SOLO si supera el mejor valor persistido en disco
    -- BUG 1 corregido: ya no depende de memoria que se pierde al reiniciar."""
    def __init__(self, phase):
        super().__init__()
        self.phase = phase
        self.best_val_accuracy = load_state().get("best_val_accuracy", 0.0)

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        val_acc = logs.get("val_accuracy", 0.0)

        self.model.save(LATEST_MODEL_PATH)

        if val_acc > self.best_val_accuracy:
            self.best_val_accuracy = val_acc
            self.model.save(BEST_MODEL_PATH)
            print(f"[checkpoint] nuevo mejor val_accuracy={val_acc:.4f}")

        save_state(self.phase, epoch + 1, self.best_val_accuracy)

resuming = os.path.exists(LATEST_MODEL_PATH)
state = load_state() if resuming else {"phase": 1, "next_epoch": 0, "best_val_accuracy": 0.0}
print("Estado:", state, "| reanudando:", resuming)

if resuming:
    model = keras.models.load_model(LATEST_MODEL_PATH)  # ya no necesita safe_mode=False
    base_model = model.layers[0]
else:

    base_model = ResNet50(
        weights='imagenet', 
        include_top=False, 
        input_shape=(*IMG_SIZE, 3)
    )

    base_model.trainable = False
    
    model = keras.Sequential([
        keras.layers.Input(shape=(*IMG_SIZE, 3)),
        base_model,
        keras.layers.GlobalAveragePooling2D(),
        keras.layers.Dense(128, activation='relu'),
        keras.layers.Dropout(0.4),
        keras.layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])

# ============== FASE 1 ==============
if state["phase"] == 1:
    state_checkpoint = StateCheckpoint(phase=1)
    history1 = model.fit(
        train_ds, validation_data=val_ds,
        initial_epoch=state["next_epoch"], epochs=EPOCHS_TRAIN_1,
        steps_per_epoch=steps_per_epoch,
        callbacks=[
            EarlyStopping(monitor='val_loss', patience=PATIENCE, restore_best_weights=True),
            ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=PATIENCE, min_lr=1e-7),
            state_checkpoint
        ],
        verbose=1
    )
    current_state = load_state()
    save_state(phase=2, next_epoch=0, best_val_accuracy=current_state["best_val_accuracy"])
    state = load_state()

# ============== FASE 2 ==============
if state["phase"] == 2:
    if state["next_epoch"] == 0:
        model.compile(optimizer=Adam(learning_rate=1e-5),
                       loss='binary_crossentropy', metrics=['accuracy'])
        base_model.trainable = True
        for layer in base_model.layers:
            if isinstance(layer, keras.layers.BatchNormalization):
                layer.trainable = False
        for layer in base_model.layers[:-50]:
            layer.trainable = False

    state_checkpoint = StateCheckpoint(phase=2)
    history2 = model.fit(
        train_ds, validation_data=val_ds,
        initial_epoch=state["next_epoch"], epochs=EPOCHS_TRAIN_2,
        steps_per_epoch=steps_per_epoch,
        callbacks=[
            EarlyStopping(monitor='val_loss', patience=PATIENCE, restore_best_weights=True),
            ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=PATIENCE, min_lr=1e-7),
            state_checkpoint
        ],
        verbose=1
    )

In [ ]:
model.save('/content/drive/MyDrive/final_model_v5.keras')
print("Entrenamiento completo. Modelo final guardado.")

In [ ]:
import numpy as np
from sklearn.metrics import balanced_accuracy_score, f1_score

thresholds_to_test = np.arange(0.3, 0.85, 0.05)
results = []

for t in thresholds_to_test:
    y_pred_t = (y_pred_probs > t).astype(int)
    results.append({
        "threshold": round(t, 2),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred_t),
        "f1_real": f1_score(y_true, y_pred_t, pos_label=1),
        "f1_ai": f1_score(y_true, y_pred_t, pos_label=0),
        "total_errors": np.sum(y_pred_t != y_true)
    })

import pandas as pd
df = pd.DataFrame(results)
print(df.to_string(index=False))

In [ ]:
df['total_errors'].min()

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import recall_score, precision_score

thresholds_to_test = np.arange(0.30, 0.75, 0.05)
results = []

for t in thresholds_to_test:
    y_pred_t = (y_pred_probs > t).astype(int)
    results.append({
        "threshold": round(t, 2),
        "recall_ai_images": recall_score(y_true, y_pred_t, pos_label=0),   # lo que más te importa
        "precision_ai_images": precision_score(y_true, y_pred_t, pos_label=0),
        "real_marcadas_como_ai": np.sum((y_true == 1) & (y_pred_t == 0)),  # costo del trade-off
    })

df = pd.DataFrame(results)
print(df.to_string(index=False))

In [ ]:
y_pred_probs = model.predict(val_ds).ravel()
y_pred_classes = (y_pred_probs > 0.55).astype(int)  # 0 = ai_images, 1 = real
y_true = np.concatenate([y for x, y in val_ds], axis=0)

In [ ]:
import matplotlib.pyplot as plt

ai_probs = y_pred_probs[y_true == 0]  # probabilidades P(real) para imágenes que SÍ son ai_images
real_probs = y_pred_probs[y_true == 1]

plt.hist(ai_probs, bins=50, alpha=0.6, label='ai_images (verdadero)')
plt.hist(real_probs, bins=50, alpha=0.6, label='real (verdadero)')
plt.axvline(0.5, color='gray', linestyle='--')
plt.axvline(0.55, color='red', linestyle='--')
plt.xlabel('P(real) predicha')
plt.legend()
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_true, y_pred_classes)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels= class_names)
disp.plot(cmap="Blues")


In [ ]:
import matplotlib.pyplot as plt

idx = 0

for images, labels in val_ds:
    batch_size = images.shape[0]

    for i in range(batch_size):
        real = y_true[idx]
        pred = y_pred_classes[idx]

        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(f"Real: {real} | Predicho: {pred}")
        plt.axis("off")
        plt.show()

        idx += 1